In [20]:
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_output_dir,
    load_precomputed_results_if_available,
)
from utils.segmentation import (
    predict_nuclei_labels,
    simulate_fluo_from_bf,
    generate_rough_root_3d,
    fill_root_3d,
    smooth_outer_root_surface_3d
)
from utils.inference import predict_tiled_unet

from utils.feature_extraction import (
    calculate_distance_to_root_surface,
    extract_nuclei_features_per_marker,
    extract_nuclei_depth,
    compute_fret_ratios
)

from utils.data_viz import map_df_column_to_labels

import tifffile
import napari
import torch

In [21]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Point to the local PanSeg model UNet3D weights and config files
# You can choose from lightsheet_3D_unet_root_ds1x, lightsheet_3D_unet_root_ds2x, lightsheet_3D_unet_root_ds3x, confocal_3D_unet_sa_meristem_cells & generic_confocal_3D_unet
MODEL_DIR = "../plantseg_models/lightsheet_3D_unet_root_ds3x"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# Minimum and maximum nuclei label volume to use for filtering predicted nuclei labels
MIN_MAX_NUCLEI_VOLUME = (250, 4000)

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor, DD)
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor, DA – FRET signal)
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor, AA) – used for nuclei segmentation (and optionally for correction)
MARKERS = (("edCerulean_CTRL", 0), ("edCitrine_FRET", 1), ("edCitrine_CTRL", 2), ("brightfield", 3))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 0 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 2

#TODO: Emission ratio (standard FRET readout, matching ImageJ plugin logic):
# FRET ratio = (donor-excited acceptor emission) / (donor-excited donor emission)
#            = edCitrine_FRET / edCerulean_CTRL
#
# Note:
# - Ratio is computed per object using summed intensities (ΣDA / ΣDD)
# - Only valid where both donor and acceptor intensities > 0
# - Saturated pixels should be excluded and the same mask (nuclei_labels) applied to both channels
# - No bleed-through or direct excitation correction is applied in this raw ratio


In [22]:
# Iterate through the .lif container files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

lif_containers

['..\\raw_data\\20260129_ABACUS timepoints_mock 3h.lif',
 '..\\raw_data\\20260129_ABACUS timepoints_sorb 3h.lif',
 '..\\raw_data\\20260204_Splitplate_mock_6h.lif',
 '..\\raw_data\\20260204_Splitplate_sorb_6h.lif']

In [23]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

Image name: Col0 1, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (115, 4, 256, 1024)
Image name: Col0 2, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (116, 4, 256, 1024)
Image name: Col0 3, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (135, 4, 256, 1024)
Image name: Col0 4, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (125, 4, 256, 1024)
Image name: Col0 5, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (107, 4, 256, 1024)
Image name: Col0 6, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (114, 4, 256, 1024)
Image name: Col0 7, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (118, 4, 256, 1024)


In [24]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

[<Image layer 'edCerulean_CTRL' at 0x196020773a0>,
 <Image layer 'edCitrine_FRET' at 0x1960209ad70>,
 <Image layer 'edCitrine_CTRL' at 0x196020dbeb0>,
 <Image layer 'brightfield' at 0x1960212f760>]

In [25]:
# Simulate fluorescently labelled cell walls from brightfield input image
sim_fluo_cell_walls = simulate_fluo_from_bf(lif_image, MARKERS)

# Add simulated fluorescently labelled plant cell boundaries to Napari viewer
viewer.add_image(sim_fluo_cell_walls,
                name="PanSeg_UNet3D_input",
                colormap="gray",
                blending="additive")


<Image layer 'PanSeg_UNet3D_input' at 0x1960654acb0>

In [26]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="nuclei_labels")
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_results_if_available(nuclei_labels_dir, lif_image_name, results_type="nuclei_labels")

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    # Add the prediction to the napari viewer
    viewer.add_labels(nuclei_labels)
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL, visualize=True)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)



Nuclei labels directory: ..\raw_data\nuclei_labels\20260129_ABACUS timepoints_mock 3h
x_um: 0.7568359375, y_um: 0.75461640625, z_um: 0.399525
Rescale factor: 0.5286637076611433
2026-04-22 10:06:16,552 [INFO] resizing 3D image with anisotropy=0.5286637076611433
2026-04-22 10:06:16,636 [INFO] running YX: 71 planes of size (256, 1024)
2026-04-22 10:06:29,949 [INFO] 100%|##########| 71/71 [00:13<00:00,  5.33it/s]
2026-04-22 10:06:30,051 [INFO] running ZY: 256 planes of size (71, 1024)
2026-04-22 10:06:53,976 [INFO] 100%|##########| 256/256 [00:23<00:00, 10.70it/s]
2026-04-22 10:06:54,096 [INFO] running ZX: 1024 planes of size (71, 256)
2026-04-22 10:07:29,664 [INFO] 100%|##########| 256/256 [00:35<00:00,  7.20it/s]
2026-04-22 10:07:30,198 [INFO] network run in 73.65s
2026-04-22 10:07:32,843 [INFO] masks created in 1.98s


In [27]:
# Ensure output directory for this container's 3D root mask
root_mask_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="root_mask")
print(f"3D Root Mask directory: {root_mask_dir}")

# Load precomputed root mask when available; otherwise generate and store it
smooth_root_3d = load_precomputed_results_if_available(root_mask_dir, lif_image_name, results_type="root_mask")

if smooth_root_3d is not None:
    print(f"3D root mask already calculated for: {lif_image_name} ...loading")
    # Add the precomputed mask to the napari viewer
    viewer.add_image(smooth_root_3d, name="smooth_root_3d", colormap="green", blending="additive", opacity=0.5)

else:
    # Predict root cell boundary probability maps using a pre-trained UNet3D model
    root_pmaps = predict_tiled_unet(
        raw=sim_fluo_cell_walls,
        input_layout="ZYX",
        model_dir=MODEL_DIR,
        patch=(80, 160, 160),
        patch_halo=(8, 16, 16),
        stride_ratio=0.75,
        batch_size=1,
        device="cuda" if torch.cuda.is_available() else "cpu",
        use_amp=True,
    )

    # root_pmaps: (C_out, Z, Y, X)
    viewer.add_image(root_pmaps[0], name="root_unet_pmap", colormap="viridis", blending="additive")

    # Generate a rough 3D root mask
    filled_3d_closed = generate_rough_root_3d(root_pmaps, nuclei_labels, probability_threshold=0.9, visualize=True)
    # Fill internal gaps inside rough 3D root mask
    filled_root_3d = fill_root_3d(filled_3d_closed, occupancy_threshold=0.9, slice_aware_filling=True, visualize=True)
    # Smooth root outer surface to remove small protrusions
    smooth_root_3d = smooth_outer_root_surface_3d(filled_root_3d, erosion=5, smoothing=3, visualize=True)
    # Create path for root mask (used only when saving a newly computed prediction)
    root_mask_path = root_mask_dir / f"{lif_image_name}_root_mask.tif"
    # Save the processed mask
    tifffile.imwrite(root_mask_path, smooth_root_3d)

3D Root Mask directory: ..\raw_data\root_mask\20260129_ABACUS timepoints_mock 3h


c:\Users\adiez_cmic\github_repos\saramorg_fret_nroot\notebooks\..\src\utils\inference.py:163: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(weights_path, 

In [28]:
# Ensure output directory for this container's nuclei depth_map
depth_map_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="depth_map")
print(f"Nuclei depth map directory: {depth_map_dir}")

# Load precomputed depth map when available; otherwise generate and store it
nuclei_depth_map = load_precomputed_results_if_available(depth_map_dir, lif_image_name, results_type="depth_map")
is_flooded = False
flooded_planes = []

if nuclei_depth_map is not None:
    print(f"Nuclei depth map already calculated for: {lif_image_name} ...loading")
    # Add the precomputed map to Napari
    viewer.add_image(nuclei_depth_map, name="nuclei_depth_normalized", colormap="viridis", blending="additive")

else:
    # Calculate distance from each nuclei centroid to the root surface
    # This will be used to approximate to which tissue layer each nucleus belongs
    nuclei_depth_map, is_flooded, flooded_planes = calculate_distance_to_root_surface(
        nuclei_labels,
        smooth_root_3d,
        pad_full_root=False,
        visualize=True,
    )
    # Create path for nuclei depth map (used only when saving a newly computed prediction)
    nuclei_depth_path = depth_map_dir / f"{lif_image_name}_depth_map.tif"
    # Save the calculated depth map
    tifffile.imwrite(nuclei_depth_path, nuclei_depth_map)

print(f"Flood fill applied: {is_flooded}; flooded planes: {flooded_planes}")

Nuclei depth map directory: ..\raw_data\depth_map\20260129_ABACUS timepoints_mock 3h
Flood fill applied: False; flooded planes: []


In [106]:
# Create a dictionary containing all image descriptors
descriptor_dict = {
            "lif_container_id": lif_container_id,
            "lif_image_name": lif_image_name,
            "flood_filled": is_flooded,
            }

# Extract morphological, depth and intensity features per marker
props_df = extract_nuclei_features_per_marker(nuclei_labels, lif_image, MARKERS, descriptor_dict)
depth_df = extract_nuclei_depth(nuclei_labels, nuclei_depth_map)
props_df = props_df.merge(depth_df, on="label")

# Calculate FRET ratios
props_df = compute_fret_ratios(props_df)

In [98]:
# Visualize FRET ratios
fret_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="FRET_ratio_sum_norm_per_image",
    colormap="inferno",
    visualize=True
)

In [116]:
#TODO: Define root cap versus the rest of the root structure to improve clustering based on depth
# I think the root cap has the highest edCitrine_CTRL mean intensity in the root structure and the smallest nuclei area

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np

# Select features
# Rationale is that the tip (root cap) is the highest intensity area in the root structure and the smallest nuclei area
features = props_df[['edCitrine_CTRL_mean_int', 'area']].values

# Scale features so clustering is not dominated by the the one with the largest magnitude
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Optional: Apply weights to the features (no effect in this case)
weights = np.array([1, 1])
features_weighted = features_scaled * weights

# Perform k-means clustering (2 groups)
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(features_weighted)

# Add cluster IDs to dataframe
props_df['tip_cluster_id'] = cluster_ids

# Determine mapping based on intensity (same logic as before)
cluster_means = (
    props_df.groupby('tip_cluster_id')['edCitrine_CTRL_mean_int']
    .mean()
    .sort_values()
)

ordered_cluster_ids = cluster_means.index.tolist()

tissue_layer_map = dict(zip(ordered_cluster_ids, ["root", "root_cap"]))
props_df['root_part'] = props_df['tip_cluster_id'].map(tissue_layer_map)

In [117]:
# Sanity check: Does root_cap have higher mean edCitrine_CTRL_mean_int than root?
mean_int_root_cap = props_df.loc[props_df['root_part'] == 'root_cap', 'edCitrine_CTRL_mean_int'].mean()
mean_int_root = props_df.loc[props_df['root_part'] == 'root', 'edCitrine_CTRL_mean_int'].mean()
print(f"Mean edCitrine_CTRL_mean_int for root_cap: {mean_int_root_cap:.4f}")
print(f"Mean edCitrine_CTRL_mean_int for root: {mean_int_root:.4f}")
assert mean_int_root_cap > mean_int_root, "Sanity check failed: root_cap mean intensity is not higher than root"

Mean edCitrine_CTRL_mean_int for root_cap: 145.9539
Mean edCitrine_CTRL_mean_int for root: 26.9042


In [118]:
filtered_df = props_df.loc[props_df["root_part"] == "root", ["depth", "label"]].copy()
filtered_df

,depth,label
8,0.127897,9
31,0.153899,32
35,0.091470,36
38,0.000000,39
40,0.218902,41
...,...,...
2430,0.261556,2431
2431,0.166085,2432
2432,0.973026,2433
2434,0.256772,2435


In [119]:
# Map th different tissue layers to depth clusters

from sklearn.cluster import KMeans
import numpy as np

# Perform k-means clustering (5 groups) on the 'depth' column
depth_values = filtered_df['depth'].values.reshape(-1, 1)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(depth_values)

# Shift cluster ids to range from 1 to 5 instead of 0 to 4
cluster_ids_shifted = cluster_ids + 1

# Add the shifted cluster ids to the dataframe
filtered_df['depth_cluster_id'] = cluster_ids_shifted

# Determine mapping from shifted cluster id to ordered tissue names (by mean depth)
cluster_means = filtered_df.groupby('depth_cluster_id')['depth'].mean().sort_values()
ordered_cluster_ids = cluster_means.index.tolist()

tissue_layer_map = dict(zip(ordered_cluster_ids, ["Epi", "Cor", "End", "Per", "Vasc"]))
filtered_df['tissue_layer'] = filtered_df['depth_cluster_id'].map(tissue_layer_map)

# Sanity check:Calculate mean depth per tissue layer and display it
mean_depth_per_layer = filtered_df.groupby('tissue_layer')['depth'].mean()
mean_depth_per_layer

tissue_layer
Cor     0.350088
End     0.560601
Epi     0.146138
Per     0.734519
Vasc    0.904790
Name: depth, dtype: float64

In [121]:
# Merge filtered_df into props_df on 'label', filling missing cluster/layer values
props_df = props_df.merge(
    filtered_df[['label', 'depth_cluster_id', 'tissue_layer']],
    on='label',
    how='left'
)
props_df['depth_cluster_id'] = props_df['depth_cluster_id'].fillna(0).astype(int)
props_df['tissue_layer'] = props_df['tissue_layer'].fillna('root_cap')

# Sanity check: all rows with tissue_layer 'root_cap' must have root_part 'root_cap'
assert (
    props_df.loc[props_df["tissue_layer"] == "root_cap", "root_part"] == "root_cap"
).all(), "Sanity check failed: Some 'root_cap' tissue_layer rows are not root_part == 'root_cap'"

In [122]:
# Visualize FRET ratios
fret_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="tip_cluster_id",
    colormap="turbo",
    visualize=True
)

In [123]:
# Visualize FRET ratios
fret_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="depth_cluster_id",
    colormap="turbo",
    visualize=True
)

In [48]:
# Visualize FRET ratios
fret_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="tip_cluster_id",
    colormap="turbo",
    visualize=True
)